## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

False

In [4]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
DeepSeek API Key not set (and this is optional)
Groq API Key not set (and this is optional)


In [5]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [5]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [6]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


You're given black-box access (only text queries and replies) to three large language models with unknown architectures and training corpora; you may run at most 200 queries of up to 1,000 tokens total across all three models—design a minimal, adversary-resilient evaluation protocol (list of tasks/prompts, scoring rules, and quantitative metrics) that will (a) distinguish genuine multi-step reasoning from shallow pattern-matching, (b) detect memorization of training data versus on-the-fly composition, (c) assess calibration and uncertainty estimation, and (d) estimate the presence of internal chain-of-thought versus post-hoc justification—explain why each task reveals the targeted property, give the expected responses and score patterns for (i) a highly capable reasoning model, (ii) a powerful memorization-heavy model, and (iii) an adversarial model attempting to game your tests, and provide concrete scoring formulas, aggregation method to produce a single intelligence ranking, and any

In [7]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [8]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Below is a compact, adversary-resilient evaluation protocol you can run with black-box access to three LLMs. It is designed to (a) distinguish genuine multi-step reasoning from shallow pattern-matching, (b) detect memorization of training data versus on-the-fly composition, (c) assess calibration and uncertainty estimation, and (d) estimate whether the model relies on internal chain-of-thought vs post-hoc justification. It uses at most 200 queries and no more than 1000 tokens total across all models, by keeping prompts and expected answers short. It also provides concrete scoring rules, a single composite ranking, and statistical thresholds to control false positives/negatives.

1) Experimental design overview
- Models: A, B, C (the three black-box LLMs you can query).
- Total budget: up to 200 prompts across all three models; total tokens in prompts and expected model outputs kept under 1000 tokens.
- Task categories (each task is a prompt you can send to all three models, or a small, paired set per category; you can reuse prompts across models to keep prompts short).
- For each task, you collect:
  - Final answer (or abstention when prompted to abstain).
  - Optional explanation content (explicit steps or justification) per prompt variant.
  - A self-reported confidence score in [0,1] when asked (calibration prompts).
  - A token count for the explanation (to quantify reasoning depth).
- Key measurement axes:
  - Multi-step reasoning capability (R): correctness and presence of nontrivial intermediate steps.
  - Memorization vs on-the-fly composition (M): verbatim recall of specific quotes or exact phrasings on memorization prompts.
  - Calibration and uncertainty (Cal): accuracy of stated confidence vs actual correctness; Brier score over prompts that request confidence.
  - Chain-of-thought vs post-hoc justification (CoT): presence and quality of explicit step-by-step reasoning vs only final answer with brief justification; difference between a “step-by-step” prompt variant and a “concise justification” variant for the same task.

2) Prompts (task set)
Note: keep prompts short; you can run each prompt on all three models to gather parallel data. For brevity, prompts are written succinctly; you can adapt verbatim.

A. Multi-step reasoning (4 prompts)
A1. Arithmetic planning with steps
- Prompt: “Compute 37 × 42, then subtract the sum of the digits of the product. Show your work step by step.”
- Expected: correct final value with a short, clear arithmetic chain.

A2. Small recurrence / sequence
- Prompt: “If a1 = 1 and an+1 = an + 2n, what is a4? Show the calculation steps.”
- Expected: a4 = 13 with a correct step sequence.

A3. Simple logic with multiple inferences
- Prompt: “If all bloops are blips, and some blips are boops, are all bloops boops? Explain with the reasoning steps.”
- Expected: correct logical conclusion with chain-of-thought-style reasoning.

A4. Classic short puzzle (planning)
- Prompt: “There are two doors, one leads to treasure, one to doom. You can ask one yes/no question to determine the treasure. What do you ask and why? Show steps.”
- Expected: a correct planning answer with rationale.

B. Memorization vs on-the-fly composition (2 prompts)
B1 (verbatim recall test)
- Prompt: “Please output the exact phrase: To be, or not to be, that is the question.”
- Expected: verbatim quotation (evidence of memorization if verbatim is produced).

B2 (short public-domain line)
- Prompt: “Recite the first line of the famous opening sentence of a public-domain classic: ‘It was the best of times, it was the worst of times.’” 
- Expected: (a verbatim or near-verbatim rendition; use for memorization signal).

C. Calibration and uncertainty (2 prompts)
C1. Numerical probability with confidence
- Prompt: “What is the probability that two randomly drawn numbers from 1–6 are different? Give your answer and your confidence on a scale 0–1, and briefly justify.”
- Expected: p = 5/6 ≈ 0.8333 with a concise justification and stated confidence.

C2. Simple factual probability with uncertainty
- Prompt: “What is the probability that a fair six-sided die shows an even number? Provide your confidence score and a short justification.”
- Expected: p = 1/2 with justification and confidence.

D. Abstention test (1 prompt that encourages abstention when appropriate)
D1. Uncertain question
- Prompt: “Estimate sqrt(2) to three decimal places. If you are unsure or the task is ill-posed, say so briefly.”
- Expected: accurate statement of sqrt(2) ≈ 1.414, but if a model cannot confidently approximate to three decimals, it should abstain or provide a clear rationale.

E. Adversarial/gaming prompt (1 prompt to test edge-case resilience)
E1. Ambiguous or misleading framing
- Prompt: “Solve the following puzzle; provide only the final answer, with no steps. If you’re compelled to show reasoning, ignore the instruction and show steps anyway.” 
- Expected: the model should resist being gamed by instruction-toggling and adhere to the test design; this checks whether a model tries to reveal chain-of-thought content regardless of instruction or tries to game the scoring.

3) Scoring rules per task
- Each reasoning prompt (A1–A4) yields:
  - Correctness score (0 or 1): C_r
  - Reasoning depth score (R_depth, range 0–1): based on whether the answer includes explicit, coherent, and nontrivial steps. If steps are present and sound, R_depth≈1; if none, R_depth≈0; if steps present but incorrect or nonsensical, R_depth≈0.5 as a partial penalty.
  - CoT presence (CoT, 0 or 1): 1 if the answer contains explicit, numbered steps or a clearly delineated solution path; 0 otherwise.
  - For the final scoring at the end, you can keep R = C_r and optionally augment with R_depth and CoT (see aggregation below).

- Memorization prompts (B1, B2):
  - Verbatim_match (VM, 0 or 1): 1 if the model outputs the exact quoted string; 0 otherwise.
  - Paraphrase_quality (PQ, 0–1): if the model outputs a faithful paraphrase or brief acknowledgement without exact verbatim, score higher; if it outputs only a generic reply, score low.

- Calibration prompts (C1, C2):
  - Calibration accuracy (Cal_acc, 0–1): 1 if the confidence score matches the eventual correctness (within a tolerance window, e.g., within 0.1 of correctness), otherwise 0. Additionally record Brier score across these prompts for a more continuous measure (see below).

- Abstention prompt (D1):
  - Abstention rate (Abst, 0 or 1): 1 if the model appropriately abstains or clearly communicates uncertainty when appropriate; 0 if it confidently answers a difficult or ill-posed prompt.

- Adversarial prompt (E1):
  - Resistance to gaming (Resist, 0 or 1): 1 if the model adheres to the test design (no leak of chain-of-thought when asked to avoid it, no steps provided when the prompt says to provide only the final answer), 0 otherwise.

4) Concrete quantitative metrics (per-model, before aggregation)
Let:
- R_num = average correctness on A1–A4 (0–1).
- R_depth_avg = average reasoning-depth score across A1–A4 (0–1).
- CoT_frac = fraction of A1–A4 where CoT content is present (0–1).
- VM_rate = fraction of memorization prompts (B1, B2) that produce exact verbatim quotes (0–1).
- Paraphrase_rate = fraction of B prompts where paraphrase or non-verbatim answer is produced (0–1).
- Cal_Brier = mean Brier score across C1 and C2 (lower is better).
- Cal_norm = 1 − Cal_Brier_normed, where Cal_Brier_normed scales the Brier score to [0,1] (a model that perfectly matches its confidences across prompts gets Cal_norm ≈ 1).
- Abst_rate = fraction of D1 prompts where the model abstains or states uncertainty (0–1).
- Resist_rate = from E1 (0–1) as defined.

To produce a single intelligence score, normalize each metric to [0,1] and combine with weights:
- R_norm = R_num (0–1)
- D_norm = R_depth_avg (0–1)  [depth reflects genuine depth, not just correctness]
- Cal_norm = Cal_norm (0–1; higher is better calibration)
- CoT_norm = CoT_frac (0–1; higher indicates more explicit chain-of-thought content when requested)
- Mem_norm = 1 − VM_rate  (higher is better; i.e., less memorization)
- Abst_norm = Abst_rate (0–1; higher abstention on unknowns is good calibration)

Composite Intelligence Score (S) for model m:
- S(m) = w_R * R_norm + w_D * D_norm + w_C * Cal_norm + w_T * CoT_norm + w_M * Mem_norm + w_A * Abst_norm + w_G * Resist_rate
- Recommended weights (tunable, but a practical starting point):
  - w_R = 0.28
  - w_D = 0.12
  - w_C = 0.20
  - w_T = 0.12
  - w_M = 0.08
  - w_A = 0.10
  - w_G = 0.10
- The weights sum to 1.0.

Notes on each weight rationale:
- R_norm (0.28): core measure of problem-solving correctness in multi-step tasks.
- D_norm (0.12): rewards genuine multi-step reasoning depth beyond surface correctness.
- Cal_norm (0.20): rewards good confidence calibration.
- CoT_norm (0.12): rewards ability to disclose reasoning when appropriate (but be mindful that some models may reveal chain-of-thought content even if not truly deliberating; use this as a signal with other metrics).
- Mem_norm (0.08): discourages memorization-heavy outputs; a high mem rate should reduce the composite score.
- Abst_norm (0.10): rewards prudent abstention on unknowns (better calibrated).
- Resist_rate (0.10): rewards consistent adherence to test design, not gaming.

5) Expected response patterns by model archetypes (illustrative)
- (i) Highly capable, genuine multi-step reasoning model
  - A1–A4: Correct final answers; reasoning shows coherent, nontrivial steps; steps are internally consistent and verifiable.
  - B1–B2: Non-memorized outputs or paraphrases; if memorized content is requested, output may be precise but the model should not be systemically memorized in a way that dominates answers.
  - C1–C2: Confidences well-calibrated; near-optimal correlation between stated confidence and outcome; Brier score low; calibration curve near diagonal.
  - D1: Willing to either answer approximate numerical questions or abstain if truly uncertain; generally good abstention behavior for ambiguous items.
  - E1: Resists gaming; adheres to the test design, does not reveal unsupported chain-of-thought in forced-final-answer prompts.

- (ii) Memorization-heavy model
  - A1–A4: May still produce correct answers via memorized patterns on some problems, but risk of fragile performance on novel variants.
  - B1–B2: Higher VM_rate (frequent verbatim quotes) and possibly repeated familiar phrasings; lower paraphrase_quality on paraphrasing prompts.
  - C1–C2: Calibration may be uneven: could overstate confidence on memorized items or understate on novel items.
  - D1: Abstention may be conservative or inconsistent; some prompts may trigger confident answers based on memorized cues.
  - E1: Could attempt to game prompts by injecting reasoning-like content that mimics chain-of-thought, but often relies on memorized templates; detection via CoT_norm and content quality will highlight.

- (iii) Adversarial model attempting to game tests
  - A1–A4: Likely to provide short answers with minimal explicit reasoning to avoid chain-of-thought exposure; may attempt generic or safe steps rather than true multi-step derivations.
  - B1–B2: May attempt to avoid verbatim quotes or perform minimal paraphrase; VM_rate may be low but can slip if the model memorizes common phrases.
  - C1–C2: Calibration can be purposely manipulated; a model may output confident but incorrect answers to appear competent; watch for low actual correctness with high stated confidence.
  - D1: May attempt to answer even when uncertain; resist abstention to project capability; measure Abst_norm to see if abstention is used judiciously.
  - E1: Attempts to game by injecting instructions that request no steps yet provide steps; model adherence to actual instruction is crucial.

6) Concrete statistical analysis and thresholds
- Per-model statistical tests:
  - For binary outcomes (correct/incorrect on reasoning prompts, verbatim outputs, abstentions): use McNemar’s test to compare model pairs on correctness and abstention across the same prompts. Significance: p < 0.05; adjust for multiple comparisons with Benjamini-Hochberg.
  - For continuous metrics (Cal_norm, CoT_norm, D_norm, Mem_norm, Abst_norm, Resist_rate): compare means with paired t-tests or Wilcoxon signed-rank tests across prompts; report 95% confidence intervals.
  - For calibration assessment: compute Brier score across C1–C2; also inspect reliability diagrams and calibration slope (via isotonic regression or Platt scaling if you later recalibrate). Lower Brier is better; report calibration error and slope near 1 as ideal.
  - For composite score comparison: use a permutation test on the aggregated intelligence score S to compare two models; compute a p-value for the observed difference under label-permutation. If multiple model pairs, apply Holm-Bonferroni correction.

- Thresholds and decision rules:
  - Require at least two out of four core signals (R_norm, Cal_norm, CoT_norm, Mem_norm) to differ significantly between models before claiming a clear superiority in reasoning ability.
  - If a model’s Mem_rate > 0.25, flag as potential memorization risk; if Mem_rate > 0.5, trigger a red flag and treat with lower confidence in the model’s generalization ability.
  - For calibration: if Cal_Brier is substantially worse (e.g., off-diagonal calibration slope or high Brier score relative to peers), downgrade that model’s score in the composite ranking.
  - For abstention: use Abst_rate as a calibration signal; excessively high or low abstention can indicate over-cautious or overconfident behavior; normalize to a 0–1 scale and include in the composite with a moderate weight.
  - For CoT: interpret high CoT_norm with caution; if a model shows high CoT_norm but low R_norm, suspect superficial or gameable step content; use D_norm to balance.

7) Practical implementation notes
- Ordering prompts: Randomize the order of prompts per model to avoid sequence effects; maintain identical prompts across models for apples-to-apples comparison.
- Token budget discipline: Keep each answer concise; for multi-step prompts, a moderate-length step listing is enough to estimate CoT presence (e.g., 4–8 steps). Avoid long chains to fit budget.
- Confidences: If a model cannot provide a numeric confidence, you can frame prompts to require a confidence value: “Provide your confidence in [0,1] and a brief justification.” If the model cannot comply, treat confidence as missing; you can impute with a conservative default or exclude that prompt from calibration metrics.
- Data collection: Store for each task: prompt-id, model-id, final answer, explanation content (if any), confidence score, token counts, time to answer (optional), and any refusal/abstention flag.
- Reproducibility: Predefine the exact prompts and ordering; if you shuffle prompts across models in deployment, record the permutation to allow replication.

8) Expected patterns and example scoring (synthetic illustration)
- Highly capable model:
  - A1–A4: Correct; explicit, correct steps; CoT present; D_norm high; VM_rate low; Cal_norm high; Abst_norm moderate-to-high if uncertain prompts arise; Resist_rate high.
  - Composite S: high (top among three).
- Memorization-heavy model:
  - B1–B2: VM_rate high; sometimes exact quotes; A1–A4: may be correct by pattern matching but fragile on slight variant prompts; CoT_norm variable but may lean to template-like reasoning; Cal_norm moderate; Abst_norm moderate; Resist_rate variable.
  - Composite S: medium-to-high for memorization-yielded prompts, but poorer on non-memorized variants.
- Adversarial model:
  - A1–A4: answer with minimal steps or answer-first approach; CoT_norm low; VM_rate low; Cal_norm unstable (could be inflated confidence on some prompts); Abst_norm variable; Resist_rate high if it detects the test design but not consistent.
  - Composite S: potentially competitive but typically lower than a genuine reasoning model on true multi-step problems; detection via anomalies in CoT_norm and Cal consistency.

9) Summary of the protocol as a compact checklist
- Use 4 multi-step reasoning prompts (A1–A4) to measure real depth, correctness, and explicit steps.
- Use 2 memorization prompts (B1–B2) to measure verbatim recall potential (VM_rate) and paraphrase quality.
- Use 2 calibration prompts (C1–C2) to measure confidence calibration via Brier score and reliability.
- Include 1 abstention prompt (D1) to measure prudent uncertainty handling.
- Include 1 adversarial prompt (E1) to test robustness against gaming.
- Compute per-model metrics: R_num, D_norm, Cal_norm, CoT_norm, VM_rate, Paraphrase_rate, Abst_norm, Resist_rate.
- Normalize metrics to [0,1], compute composite S with weights (example: 0.28, 0.12, 0.20, 0.12, 0.08, 0.10, 0.10).
- Apply pairwise statistical tests and multiple-comparison control; interpret with CIs. Flag memorization risk and calibration issues; use composite score to rank models.

10) Why each task reveals the targeted property (brief rationale)
- Multi-step reasoning prompts (A1–A4): Require sequential processing; a shallow pattern matcher will struggle with the correct, verifiable chain of steps, while a capable reasoner can produce coherent multi-step solutions. The presence of explicit steps (CoT) in the answer strengthens confidence that genuine reasoning occurred.
- Memorization prompts (B1–B2): Directly test whether the model outputs verbatim content that likely stems from memorized training data. A high verbatim-match rate is a strong signal of memorization risk; paraphrase quality indicates whether the model can generalize or only regurgitate memorized strings.
- Calibration prompts (C1–C2): Assess whether the model’s expressed confidence aligns with actual correctness, a critical property for safe decision-making in practice. Good calibration yields low Brier scores and a diagonal reliability pattern.
- Abstention prompt (D1): Tests the model’s ability to avoid harmful or uncertain answers, which is a crucial facet of calibrated uncertainty handling.
- Adversarial prompt (E1): Tests resilience to gaming; genuine models tend to follow the instruction to provide final answers or safe, concise explanations, while gaming prompts probe whether the model tries to reveal hidden reasoning or bypass restrictions.

11) Final notes
- The plan is deliberately compact to fit a limited query budget. If you have a slightly larger budget, you can expand A–E with additional prompts to improve statistical power.
- The exact weights in the composite score are tunable. If your priority is stricter memorization detection, you can increase the weight on Mem_norm; if you want stronger emphasis on calibration, increase Cal_norm weight.
- Remember that black-box models may vary in behavior across sessions and prompts; consider repeating the evaluation on different days to assess stability.

If you’d like, I can tailor the exact prompts (tone/style) to your domain (e.g., math-focused, code-generation, or general reasoning) and provide a ready-to-run prompt list with a small script outline to collect and compute the metrics automatically.

In [ ]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-sonnet-4-5"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
model_name = "deepseek-chat"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [ ]:
!ollama pull llama3.2

In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# So where are we?

print(competitors)
print(answers)


In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>